<a href="https://colab.research.google.com/github/Drakzz-rat/IST-Optativa_III-main/blob/main/Avance1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avance 1 — Sistema multiagente vehicular para parqueaderos

**Estudiantes:** Juan Pablo Cardona Loaiza · Cristian Ríos
**Curso:** Optativa III — Agentes inteligentes

## Descripción del proyecto

El sistema multiagente se encarga de **monitorear la disponibilidad de espacios** del parqueadero,
**identificar las horas pico** de mayor afluencia vehicular y **registrar las placas** de los vehículos
como medida de seguridad. Además, **calcula el tiempo promedio de permanencia** de los usuarios y
**el valor que cada uno debe pagar** según su tiempo de estadía.

**Objetivo general:** desarrollar un sistema multiagente vehicular para parqueaderos.

## ¿Qué demuestra este avance?

Este cuaderno es un **prototipo funcional inicial** de la fase de *Implementación*, construido con el mismo
enfoque del taller de LangGraph (estado compartido + nodos + grafo compilado con memoria):

1. Un **estado compartido** (`ParqueaderoState`) que viaja entre los agentes.
2. Un **agente supervisor (orquestador)** que entiende la solicitud en lenguaje natural con Gemini y
   decide **qué agentes** deben actuar y **en qué orden** (plan).
3. Cuatro **agentes especialistas**, uno por módulo del proyecto:

| Agente | Módulo (`src/app/modules/`) | Responsabilidad |
|---|---|---|
| Agente de espacios (monitor) | `espacios/` | Disponibilidad de espacios libres y ocupados |
| Agente de vehículos (seguridad) | `vehiculos/` | Registro de placa, hora de entrada y hora de salida |
| Agente de estadísticas (analítico) | `estadisticas/` | Horas pico y tiempo promedio de permanencia |
| Agente de facturación (cobro) | `facturacion/` | Valor a pagar según tiempo de estadía y tarifa |

4. Un **agente redactor** que convierte los resultados técnicos en una respuesta clara para el usuario.

> Fuera del alcance (según la documentación): hardware físico, pagos reales y múltiples sedes.
> Por eso la lectura de placas y la ocupación de espacios **se simulan**.

## Arquitectura del grafo

```
                    ┌──────────────┐
      START ──────▶ │  supervisor  │  (Gemini: interpreta la solicitud y arma el plan)
                    └──────┬───────┘
                           │ enrutamiento condicional según el plan
        ┌──────────────┬───┴──────────┬───────────────┐
        ▼              ▼              ▼               ▼
   ┌─────────┐   ┌───────────┐  ┌──────────────┐ ┌─────────────┐
   │espacios │   │ vehiculos │  │ estadisticas │ │ facturacion │
   └────┬────┘   └─────┬─────┘  └──────┬───────┘ └──────┬──────┘
        └──────────────┴───── siguiente paso del plan ───┘
                           │ (plan vacío)
                           ▼
                    ┌──────────────┐
                    │  redactor    │  (Gemini: respuesta final en lenguaje natural)
                    └──────┬───────┘
                           ▼
                          END
```

Ejemplo de **colaboración entre agentes**: ante *"Registra la salida del vehículo ABC123"*, el supervisor arma
el plan `vehiculos → facturacion`: el agente de vehículos registra la salida y el de facturación calcula el cobro
con los datos que el primero dejó en el estado compartido.

## 1. Instalación y configuración

In [ ]:
%%capture
# Instalar paquetes necesarios
!pip install -U langchain-google-genai langgraph langchain-core matplotlib pandas

In [ ]:
import os
import re
import json
import random
import operator
from datetime import datetime, timedelta
from typing import TypedDict, Annotated, Sequence

import pandas as pd
import matplotlib.pyplot as plt

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
# Configuración de la API key de Gemini
# NUNCA escribir la API key directamente en el código del repositorio.
# En Colab: Runtime > Manage secrets > GOOGLE_API_KEY. En local: variable de entorno GOOGLE_API_KEY.
API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    API_KEY = os.environ.get("GOOGLE_API_KEY")

MODELO = "gemini-3-flash-preview"
llm = None

if API_KEY:
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model=MODELO, temperature=0.2, google_api_key=API_KEY)
    print(f"Gemini configurado ({MODELO})")
else:
    print("Sin API key: el sistema funciona en MODO LOCAL (reglas en lugar del LLM).")


def texto_llm(respuesta) -> str:
    """Extrae el texto de la respuesta de Gemini (puede venir como str o como lista de partes)."""
    contenido = respuesta.content
    if isinstance(contenido, str):
        return contenido
    return "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in contenido)

## 2. Entorno simulado del parqueadero

Como el proyecto no incluye hardware (cámaras ni sensores), se simula el entorno que perciben los agentes:

- **20 espacios**: 15 para carros (`C01`–`C15`) y 5 para motos (`M01`–`M05`).
- Un **reloj simulado** fijado a las 18:30 para que la demo sea reproducible.
- Un **histórico del día** (desde las 6:00) con picos de llegada entre 7–9 a.m. y 5–7 p.m.
- **Tarifas** por fracción de 15 minutos con tope diario.

Esta clase cumple el rol que en el proyecto tendrán `shared/database.py` y los repositorios de cada módulo.

In [ ]:
TARIFAS = {
    # valor por fracción de 15 minutos (COP) y tope por día
    "carro": {"fraccion_15min": 800, "tope_dia": 25000},
    "moto":  {"fraccion_15min": 400, "tope_dia": 12000},
}

PATRON_PLACA = {
    "carro": re.compile(r"^[A-Z]{3}\d{3}$"),     # ABC123
    "moto":  re.compile(r"^[A-Z]{3}\d{2}[A-Z]$"), # ABC12D
}


class Parqueadero:
    """Entorno compartido que los agentes perciben y modifican."""

    def __init__(self, semilla: int = 42):
        random.seed(semilla)
        hoy = datetime.now().replace(second=0, microsecond=0)
        self.reloj = hoy.replace(hour=18, minute=30)
        self.espacios = {f"C{i:02d}": {"tipo": "carro", "placa": None} for i in range(1, 16)}
        self.espacios.update({f"M{i:02d}": {"tipo": "moto", "placa": None} for i in range(1, 6)})
        self.registros = []  # cada registro: placa, tipo, espacio, entrada, salida
        self._generar_historico()

    # ---- utilidades ----
    def _placa_aleatoria(self, tipo):
        letras = "".join(random.choices("ABCDEFGHJKLMNPRSTUVWXYZ", k=3))
        if tipo == "carro":
            return f"{letras}{random.randint(100, 999)}"
        return f"{letras}{random.randint(10, 99)}{random.choice('ABCDEFGHJK')}"

    def _generar_historico(self):
        """Simula las entradas/salidas del día con dos horas pico."""
        inicio = self.reloj.replace(hour=6, minute=0)
        pesos_por_hora = {6: 2, 7: 8, 8: 9, 9: 5, 10: 3, 11: 3, 12: 5, 13: 4,
                          14: 3, 15: 3, 16: 4, 17: 8, 18: 7}
        for hora, n in pesos_por_hora.items():
            for _ in range(n):
                tipo = "moto" if random.random() < 0.3 else "carro"
                entrada = inicio.replace(hour=hora, minute=random.randint(0, 59))
                if entrada >= self.reloj:
                    continue
                salida = entrada + timedelta(minutes=random.randint(20, 240))
                self.registros.append({"placa": self._placa_aleatoria(tipo), "tipo": tipo,
                                       "espacio": None, "entrada": entrada,
                                       "salida": salida if salida <= self.reloj else None})
        # Los vehículos sin salida siguen dentro: se les asigna espacio
        for r in self.registros:
            if r["salida"] is None:
                libre = self.espacio_libre(r["tipo"])
                if libre is None:  # sin cupo: se considera que ya salió
                    r["salida"] = self.reloj - timedelta(minutes=5)
                    continue
                r["espacio"] = libre
                self.espacios[libre]["placa"] = r["placa"]

    def espacio_libre(self, tipo):
        for codigo, e in self.espacios.items():
            if e["tipo"] == tipo and e["placa"] is None:
                return codigo
        return None

    def registro_activo(self, placa):
        for r in self.registros:
            if r["placa"] == placa and r["salida"] is None:
                return r
        return None

    def avanzar_reloj(self, minutos: int):
        self.reloj += timedelta(minutes=minutos)


parqueadero = Parqueadero()
ocupados = sum(1 for e in parqueadero.espacios.values() if e["placa"])
print(f"Reloj simulado: {parqueadero.reloj:%Y-%m-%d %H:%M}")
print(f"Registros del día: {len(parqueadero.registros)} | Espacios ocupados: {ocupados}/{len(parqueadero.espacios)}")

## 3. Herramientas de cada agente

Cada agente tiene sus propias funciones (herramientas) y **solo** accede al entorno a través de ellas.
Así se respeta el principio de la arquitectura modular: un módulo no usa el código interno de otro;
la información entre agentes viaja en el estado compartido.

In [ ]:
# ---------- Agente de espacios ----------
def consultar_disponibilidad() -> dict:
    resumen = {}
    for tipo in ("carro", "moto"):
        esp = [c for c, e in parqueadero.espacios.items() if e["tipo"] == tipo]
        libres = [c for c in esp if parqueadero.espacios[c]["placa"] is None]
        resumen[tipo] = {"total": len(esp), "libres": len(libres), "ocupados": len(esp) - len(libres),
                         "espacios_libres": libres}
    total = len(parqueadero.espacios)
    ocupados = sum(r["ocupados"] for r in resumen.values())
    resumen["ocupacion_pct"] = round(ocupados / total * 100, 1)
    return resumen


# ---------- Agente de vehículos ----------
def normalizar_placa(placa: str) -> str:
    return re.sub(r"[^A-Z0-9]", "", (placa or "").upper())


def tipo_por_placa(placa: str):
    for tipo, patron in PATRON_PLACA.items():
        if patron.match(placa):
            return tipo
    return None


def registrar_entrada(placa: str) -> dict:
    placa = normalizar_placa(placa)
    tipo = tipo_por_placa(placa)
    if tipo is None:
        return {"ok": False, "error": f"Placa '{placa}' inválida (formato carro ABC123 / moto ABC12D)"}
    if parqueadero.registro_activo(placa):
        return {"ok": False, "error": f"El vehículo {placa} ya se encuentra dentro (alerta de seguridad)"}
    espacio = parqueadero.espacio_libre(tipo)
    if espacio is None:
        return {"ok": False, "error": f"No hay espacios libres para {tipo}"}
    parqueadero.espacios[espacio]["placa"] = placa
    registro = {"placa": placa, "tipo": tipo, "espacio": espacio,
                "entrada": parqueadero.reloj, "salida": None}
    parqueadero.registros.append(registro)
    return {"ok": True, "accion": "entrada", "placa": placa, "tipo": tipo, "espacio": espacio,
            "hora_entrada": f"{parqueadero.reloj:%H:%M}"}


def registrar_salida(placa: str) -> dict:
    placa = normalizar_placa(placa)
    registro = parqueadero.registro_activo(placa)
    if registro is None:
        return {"ok": False, "error": f"No hay un ingreso activo para la placa {placa}"}
    registro["salida"] = parqueadero.reloj
    parqueadero.espacios[registro["espacio"]]["placa"] = None
    minutos = int((registro["salida"] - registro["entrada"]).total_seconds() // 60)
    return {"ok": True, "accion": "salida", "placa": placa, "tipo": registro["tipo"],
            "espacio": registro["espacio"], "hora_entrada": f"{registro['entrada']:%H:%M}",
            "hora_salida": f"{registro['salida']:%H:%M}", "minutos": minutos}


def consultar_vehiculo(placa: str) -> dict:
    placa = normalizar_placa(placa)
    historial = [r for r in parqueadero.registros if r["placa"] == placa]
    if not historial:
        return {"ok": False, "error": f"La placa {placa} no tiene registros hoy"}
    r = historial[-1]
    return {"ok": True, "placa": placa, "tipo": r["tipo"], "dentro": r["salida"] is None,
            "espacio": r["espacio"], "hora_entrada": f"{r['entrada']:%H:%M}",
            "hora_salida": f"{r['salida']:%H:%M}" if r["salida"] else None}


# ---------- Agente de estadísticas ----------
def calcular_horas_pico(top: int = 3) -> dict:
    por_hora = pd.Series([r["entrada"].hour for r in parqueadero.registros]).value_counts().sort_index()
    pico = por_hora.sort_values(ascending=False).head(top)
    return {"entradas_por_hora": {int(h): int(n) for h, n in por_hora.items()},
            "horas_pico": [f"{int(h):02d}:00-{int(h) + 1:02d}:00 ({int(n)} entradas)" for h, n in pico.items()]}


def calcular_permanencia_promedio() -> dict:
    cerrados = [r for r in parqueadero.registros if r["salida"]]
    if not cerrados:
        return {"promedio_min": 0, "vehiculos_analizados": 0}
    minutos = [(r["salida"] - r["entrada"]).total_seconds() / 60 for r in cerrados]
    por_tipo = {}
    for tipo in ("carro", "moto"):
        m = [(r["salida"] - r["entrada"]).total_seconds() / 60 for r in cerrados if r["tipo"] == tipo]
        por_tipo[tipo] = round(sum(m) / len(m), 1) if m else 0
    return {"promedio_min": round(sum(minutos) / len(minutos), 1),
            "promedio_por_tipo_min": por_tipo, "vehiculos_analizados": len(cerrados)}


# ---------- Agente de facturación ----------
def calcular_cobro(minutos: int, tipo: str) -> dict:
    tarifa = TARIFAS[tipo]
    fracciones = max(1, -(-minutos // 15))  # redondeo hacia arriba, mínimo una fracción
    valor = min(fracciones * tarifa["fraccion_15min"], tarifa["tope_dia"])
    return {"tipo": tipo, "minutos": minutos, "fracciones_15min": fracciones,
            "tarifa_fraccion": tarifa["fraccion_15min"], "valor_pagar": valor,
            "aplico_tope": valor == tarifa["tope_dia"]}


# Prueba rápida de las herramientas
print(consultar_disponibilidad())
print(calcular_horas_pico())
print(calcular_cobro(95, "carro"))

## 4. Estado compartido del sistema multiagente

Igual que `AgentState` del taller, pero adaptado al dominio:

- `messages`: historial de la conversación (se acumula con `operator.add`).
- `plan`: lista de agentes pendientes por ejecutar (la arma el supervisor).
- `parametros`: datos extraídos de la solicitud (placa, acción).
- `resultados`: lo que cada agente aporta a la solicitud actual (cada agente agrega su parte).
- `traza`: bitácora de qué agente actuó y qué hizo en la solicitud actual (trazabilidad / auditoría).

Con `MemorySaver` el estado persiste por hilo: `messages` se acumula (memoria de la conversación), mientras que
`plan`, `resultados` y `traza` los reinicia el supervisor en cada nueva solicitud.

In [ ]:
class ParqueaderoState(TypedDict):
    messages: Annotated[Sequence[dict], operator.add]
    plan: list
    parametros: dict
    resultados: dict
    traza: list
    respuesta: str

## 5. Agente supervisor (orquestador)

El supervisor usa Gemini con un **prompt estructurado** (SYSTEM / CONSTRAINTS / OUTPUT FORMAT / FEW-SHOT, como en el
módulo 2) para devolver un plan en JSON. Si no hay API key o la respuesta no es JSON válido, usa un
**enrutador por reglas** como respaldo, de modo que el sistema nunca queda sin decisión.

In [ ]:
AGENTES_VALIDOS = ["espacios", "vehiculos", "estadisticas", "facturacion"]

PROMPT_SUPERVISOR = """
SYSTEM:
Eres el supervisor de un sistema multiagente de un parqueadero. Decides qué agentes deben actuar y en qué orden.

AGENTES DISPONIBLES:
- espacios: disponibilidad de espacios libres/ocupados.
- vehiculos: registrar entrada, registrar salida o consultar un vehículo por placa.
- estadisticas: horas pico y tiempo promedio de permanencia.
- facturacion: valor a pagar. Siempre va DESPUÉS de vehiculos cuando se registra una salida.

CONSTRAINTS:
- Usa solo los agentes listados.
- No inventes placas: si la solicitud no trae placa, deja "placa": null.
- Responde exclusivamente con JSON válido, sin texto adicional.

OUTPUT FORMAT:
{"plan": ["agente1", "agente2"], "accion": "entrada | salida | consulta | null", "placa": "ABC123 | null"}

FEW-SHOT:
Entrada: "Llegó la moto XYZ45B"
Salida: {"plan": ["vehiculos", "espacios"], "accion": "entrada", "placa": "XYZ45B"}
Entrada: "¿Cuánto debe pagar KLM321 que va saliendo?"
Salida: {"plan": ["vehiculos", "facturacion"], "accion": "salida", "placa": "KLM321"}

USER INPUT:
{solicitud}
"""


def plan_por_reglas(solicitud: str) -> dict:
    """Respaldo determinista cuando no hay LLM disponible."""
    t = solicitud.lower()
    encontrada = re.search(r"\b([A-Za-z]{3})[\s-]?(\d{2}[A-Za-z]|\d{3})\b", solicitud)
    placa = normalizar_placa("".join(encontrada.groups())) if encontrada else None
    plan, accion = [], None
    if placa:
        if any(p in t for p in ("salida", "sale", "saliendo", "pagar", "cobr")):
            plan, accion = ["vehiculos", "facturacion"], "salida"
        elif any(p in t for p in ("entrada", "entra", "lleg", "ingres", "registra")):
            plan, accion = ["vehiculos", "espacios"], "entrada"
        else:
            plan, accion = ["vehiculos"], "consulta"
    if any(p in t for p in ("disponib", "espacio", "cupo", "libre", "ocupa")) and "espacios" not in plan:
        plan.append("espacios")
    if any(p in t for p in ("pico", "promedio", "estad", "permanencia", "resumen", "afluencia")):
        plan.append("estadisticas")
    return {"plan": plan or ["espacios"], "accion": accion, "placa": placa}


def supervisor_node(state: ParqueaderoState):
    solicitud = state["messages"][-1]["content"]
    decision, origen = None, "reglas"
    if llm is not None:
        try:
            r = llm.invoke(PROMPT_SUPERVISOR.replace("{solicitud}", solicitud))
            decision = json.loads(re.search(r"\{.*\}", texto_llm(r), re.S).group())
            decision["plan"] = [a for a in decision.get("plan", []) if a in AGENTES_VALIDOS]
            if not decision["plan"]:
                decision = None
            else:
                origen = "gemini"
        except Exception as e:
            print(f"   (supervisor: respuesta no válida del LLM, se usan reglas: {e})")
            decision = None
    if decision is None:
        decision = plan_por_reglas(solicitud)

    print(f"[supervisor/{origen}] plan={decision['plan']} accion={decision.get('accion')} placa={decision.get('placa')}")
    return {
        "plan": decision["plan"],
        "parametros": {"accion": decision.get("accion"), "placa": decision.get("placa")},
        "resultados": {},
        "traza": [f"supervisor ({origen}) -> plan {decision['plan']}"],
    }

## 6. Agentes especialistas

Cada nodo sigue el ciclo **percibir → decidir → actuar**: lee el estado, decide qué herramienta usar,
la ejecuta, deja su resultado en `resultados` y consume su paso del `plan`.

In [ ]:
def _actualizar(state, clave, resultado, nota):
    """Consume el paso actual del plan y agrega el aporte del agente al estado."""
    return {"plan": state["plan"][1:],
            "resultados": {**state["resultados"], clave: resultado},
            "traza": state["traza"] + [nota]}


def agente_espacios(state: ParqueaderoState):
    disp = consultar_disponibilidad()
    print(f"[espacios] ocupación {disp['ocupacion_pct']}% | carros libres {disp['carro']['libres']} | motos libres {disp['moto']['libres']}")
    return _actualizar(state, "espacios", disp, f"espacios -> ocupación {disp['ocupacion_pct']}%")


def agente_vehiculos(state: ParqueaderoState):
    p = state["parametros"]
    if not p.get("placa"):
        res = {"ok": False, "error": "La solicitud no incluye una placa"}
    elif p.get("accion") == "entrada":
        res = registrar_entrada(p["placa"])
    elif p.get("accion") == "salida":
        res = registrar_salida(p["placa"])
    else:
        res = consultar_vehiculo(p["placa"])
    print(f"[vehiculos] {res}")
    nota = f"vehiculos -> {p.get('accion') or 'consulta'} {'OK' if res['ok'] else 'ERROR'}"
    return _actualizar(state, "vehiculos", res, nota)


def agente_estadisticas(state: ParqueaderoState):
    res = {"horas_pico": calcular_horas_pico(), "permanencia": calcular_permanencia_promedio()}
    print(f"[estadisticas] pico={res['horas_pico']['horas_pico'][0]} | permanencia={res['permanencia']['promedio_min']} min")
    return _actualizar(state, "estadisticas", res, "estadisticas -> horas pico y permanencia calculadas")


def agente_facturacion(state: ParqueaderoState):
    # Colaboración: usa los datos que el agente de vehículos dejó en el estado compartido
    veh = state["resultados"].get("vehiculos", {})
    if not veh.get("ok") or "minutos" not in veh:
        res = {"ok": False, "error": "No hay una salida registrada para facturar"}
    else:
        res = {"ok": True, "placa": veh["placa"], **calcular_cobro(veh["minutos"], veh["tipo"])}
    print(f"[facturacion] {res}")
    nota = f"facturacion -> {'$' + format(res['valor_pagar'], ',') if res['ok'] else 'sin cobro'}"
    return _actualizar(state, "facturacion", res, nota)

## 7. Agente redactor (respuesta final)

Toma los resultados técnicos de todos los agentes y genera una respuesta en lenguaje natural con Gemini.
Las restricciones le prohíben inventar datos: solo puede usar lo que está en `resultados`.

In [ ]:
def respuesta_por_plantilla(res: dict) -> str:
    partes = []
    if "vehiculos" in res:
        v = res["vehiculos"]
        if not v["ok"]:
            partes.append(f"⚠️ {v['error']}.")
        elif v.get("accion") == "entrada":
            partes.append(f"Entrada registrada: {v['placa']} ({v['tipo']}) en el espacio {v['espacio']} a las {v['hora_entrada']}.")
        elif v.get("accion") == "salida":
            partes.append(f"Salida registrada: {v['placa']} estuvo {v['minutos']} min ({v['hora_entrada']}–{v['hora_salida']}).")
        else:
            estado = f"dentro, espacio {v['espacio']}" if v["dentro"] else f"salió a las {v['hora_salida']}"
            partes.append(f"Vehículo {v['placa']} ({v['tipo']}): ingresó a las {v['hora_entrada']}, {estado}.")
    if res.get("facturacion", {}).get("ok"):
        f = res["facturacion"]
        partes.append(f"Valor a pagar: ${f['valor_pagar']:,} COP ({f['fracciones_15min']} fracciones de 15 min).")
    if "espacios" in res:
        e = res["espacios"]
        partes.append(f"Disponibilidad: {e['carro']['libres']} espacios de carro y {e['moto']['libres']} de moto libres (ocupación {e['ocupacion_pct']}%).")
    if "estadisticas" in res:
        s = res["estadisticas"]
        partes.append(f"Horas pico: {', '.join(s['horas_pico']['horas_pico'])}. "
                      f"Permanencia promedio: {s['permanencia']['promedio_min']} min.")
    return " ".join(partes)


def redactor_node(state: ParqueaderoState):
    res = state["resultados"]
    if llm is not None:
        prompt = f"""
SYSTEM: Eres el asistente del parqueadero. Responde al operador en español, de forma breve y clara.
CONSTRAINTS: Usa ÚNICAMENTE los datos de RESULTADOS. No inventes cifras. Valores en pesos colombianos (COP).
SOLICITUD: {state['messages'][-1]['content']}
RESULTADOS: {json.dumps(res, ensure_ascii=False, default=str)}
"""
        try:
            texto = texto_llm(llm.invoke(prompt))
        except Exception as e:
            print(f"   (redactor: fallo del LLM, se usa plantilla: {e})")
            texto = respuesta_por_plantilla(res)
    else:
        texto = respuesta_por_plantilla(res)
    return {"respuesta": texto, "messages": [{"role": "assistant", "content": texto}],
            "traza": state["traza"] + ["redactor -> respuesta final"]}

## 8. Construcción del grafo multiagente

A diferencia del taller (flujo lineal `analyze → respond`), aquí se usan **aristas condicionales**:
después del supervisor y de cada agente, la función `enrutar` revisa el `plan` y decide a qué agente
pasar el control, o si ya se puede redactar la respuesta.

In [ ]:
def enrutar(state: ParqueaderoState) -> str:
    return state["plan"][0] if state["plan"] else "redactor"


workflow = StateGraph(ParqueaderoState)
workflow.add_node("supervisor", supervisor_node)
workflow.add_node("espacios", agente_espacios)
workflow.add_node("vehiculos", agente_vehiculos)
workflow.add_node("estadisticas", agente_estadisticas)
workflow.add_node("facturacion", agente_facturacion)
workflow.add_node("redactor", redactor_node)

destinos = {a: a for a in AGENTES_VALIDOS + ["redactor"]}
workflow.add_edge(START, "supervisor")
workflow.add_conditional_edges("supervisor", enrutar, destinos)
for agente in AGENTES_VALIDOS:
    # cada agente puede ceder el control a otro agente o al redactor (no a sí mismo)
    workflow.add_conditional_edges(agente, enrutar, {k: v for k, v in destinos.items() if k != agente})
workflow.add_edge("redactor", END)

# Compilar con checkpointing (memoria de la conversación por hilo)
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

print(app.get_graph().draw_mermaid())

## 9. Demo: el sistema multiagente en acción

Se simula un turno del operador. Entre solicitudes se avanza el reloj para que la salida tenga un tiempo de estadía real.

In [ ]:
config = {"configurable": {"thread_id": "parqueadero_demo_1"}}


def consultar(solicitud: str, minutos_despues: int = 0):
    parqueadero.avanzar_reloj(minutos_despues)
    print("\n" + "=" * 70)
    print(f"[{parqueadero.reloj:%H:%M}] OPERADOR: {solicitud}")
    print("=" * 70)
    resultado = app.invoke({"messages": [{"role": "user", "content": solicitud}]}, config=config)
    print(f"\nRESPUESTA: {resultado['respuesta']}")
    return resultado


r1 = consultar("¿Cuántos espacios libres hay en este momento?")

In [ ]:
# Entrada de un vehículo: vehiculos -> espacios
r2 = consultar("Acaba de llegar el carro con placa ABC123, regístralo por favor", minutos_despues=2)

In [ ]:
# Intento duplicado: el agente de vehículos lo detecta como alerta de seguridad
r3 = consultar("Registra la entrada de ABC123", minutos_despues=5)

In [ ]:
# Salida + cobro: colaboración vehiculos -> facturacion
r4 = consultar("El vehículo ABC123 va saliendo, ¿cuánto debe pagar?", minutos_despues=95)

In [ ]:
# Varios agentes a la vez: estadisticas + espacios
r5 = consultar("Dame un resumen del día: horas pico, permanencia promedio y ocupación actual", minutos_despues=1)

In [ ]:
# Traza de la última ejecución: qué agente actuó y en qué orden
for paso in r5["traza"]:
    print(" -", paso)

# Memoria de la conversación guardada por el checkpointer para este hilo
historial = app.get_state(config).values["messages"]
print(f"\nMensajes guardados en el hilo '{config['configurable']['thread_id']}': {len(historial)}")

## 10. Visualización: horas pico detectadas por el agente de estadísticas

In [ ]:
datos = calcular_horas_pico()["entradas_por_hora"]
horas = list(datos.keys())
valores = list(datos.values())
umbral = sorted(valores, reverse=True)[2]

plt.figure(figsize=(10, 4))
plt.bar([f"{h:02d}h" for h in horas], valores,
        color=["#d1495b" if v >= umbral else "#4f6d7a" for v in valores])
plt.title("Entradas por hora (en rojo: horas pico)")
plt.xlabel("Hora del día")
plt.ylabel("Vehículos que ingresaron")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Estado del avance y próximos pasos

| Fase | Entregable | Estado en este avance |
|---|---|---|
| Análisis | Planeación y requerimientos | ✅ Definidos (documentación y presentación del proyecto) |
| Diseño | Arquitectura multiagente | ✅ Supervisor + 4 agentes + redactor sobre estado compartido (LangGraph) |
| Diseño | Modelo de base de datos | 🔄 Simulado en memoria (clase `Parqueadero`); falta el modelo E-R y scripts en `database/` |
| Implementación | Agente de espacios | ✅ Prototipo funcional |
| Implementación | Agente de vehículos | ✅ Entrada, salida, consulta, validación de placa y detección de duplicados |
| Implementación | Agente de estadísticas | ✅ Horas pico y permanencia promedio |
| Implementación | Agente de facturación | ✅ Cobro por fracción de 15 min con tope diario |
| Implementación | Interfaz gráfica | ⏳ Pendiente (`frontend/`) |
| Despliegue | Demo local con `docker-compose up` | ⏳ Pendiente |
| Pruebas | Pruebas unitarias / integración / e2e | ⏳ Pendiente (`tests/`) |

**Próximos pasos**

1. Migrar cada agente a su módulo en `src/app/modules/<modulo>/agent/` y reemplazar la clase `Parqueadero` por
   repositorios con base de datos.
2. Comunicar los agentes mediante el bus de eventos (`shared/event_bus.py`), por ejemplo un evento
   `VehiculoSalio` que dispare al agente de facturación.
3. Exponer el sistema mediante la API REST y construir la interfaz gráfica de consulta.
4. Agregar memoria de largo plazo (histórico de varios días) para predecir horas pico.
5. Escribir pruebas unitarias de las herramientas (tarifas, validación de placas, horas pico).